In [1]:
import json
import duckdb
import requests

In [2]:
db_connection = duckdb.connect('loadsmart/dev.duckdb')

In [3]:
manifest = 'loadsmart/target/manifest.json'
with open(manifest, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

In [4]:
table_content = []
for item_id, item in manifest.get('nodes', {}).items():
    if item.get('resource_type') == 'model':
        table = item.get('name')
        table_ds = item.get('description', '')
        columns = item.get('columns', {})
        columns_ds = [f" - {col_name}: {col_info.get('description', '')}" for col_name, col_info in columns.items()]
        table_content.append(f"Table: {table}\nDescription: {table_ds}\nColumns:\n" + "\n".join(columns_ds))

In [5]:
ai_guide = "\n\n".join(table_content)

In [6]:
def ask_ai_db_question(question):
    ai_prompt = f"""
        You are an expert DuckDB SQL AI.

        SCHEMA METADATA (Generated from dbt):
        {ai_guide}

        CRITICAL BEHAVIORAL RULES:
        1. You MUST read and strictly obey all `description` fields in the schema metadata above. They contain mandatory rules for handling historical dates, avoiding CURRENT_DATE, and filtering metrics.
        2. Write a valid DuckDB SQL query to answer the question.
        3. Return ONLY the executable SQL query in a markdown code block (```sql ... ```).

        Question: {question}
    """

    api_response = requests.post(
        "http://localhost:11434/api/generate",
        json = {
            'model': 'qwen2.5-coder:7b',
            'prompt': ai_prompt,
            'stream': False,
            'options': {'temperature': 0.0}
        }
    )

    llm_response = api_response.json()
    raw_sql = llm_response.get('response', '')
    cleaned_sql = raw_sql.replace('```sql', '').replace('```', '').strip()

    print(f"--- SQL ---\n{cleaned_sql}\n ---")

    # old code (no retry, crashes if the SQL is broken):
    # result_table = db_connection.execute(cleaned_sql).fetchdf()
    # return cleaned_sql, result_table

    # new code: if the SQL fails, send the error back to the model and try once more
    try:
        result_table = db_connection.execute(cleaned_sql).fetchdf()
    except Exception as e:
        fix_prompt = ai_prompt + f"\n\nThat query failed with this error:\n{e}\n\nFix it and return only the corrected SQL."
        api_response = requests.post(
            "http://localhost:11434/api/generate",
            json = {
                'model': 'qwen2.5-coder:7b',
                'prompt': fix_prompt,
                'stream': False,
                'options': {'temperature': 0.0}
            }
        )
        cleaned_sql = api_response.json().get('response', '').replace('```sql', '').replace('```', '').strip()
        print(f"--- retry SQL ---\n{cleaned_sql}\n ---")
        result_table = db_connection.execute(cleaned_sql).fetchdf()

    return cleaned_sql, result_table

In [7]:
questions = [
    "How many loads were delivered in the last full month available in the data?",
    "Which shipper had the highest total book price?",
    "What is the average book price per load by pickup state?",
    "What are the top 5 lanes by number of delivered loads?",
    "Which carrier moved the most loads into Texas?",
    "How does the average book price compare between intrastate and interstate loads?",
    "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?",
    "Among lanes with at least 10 delivered loads, which had the highest average book price?",
]

for question in questions:
    print(f"Q: {question}")
    try:
        sql_query, result_df = ask_ai_db_question(question)
        display(result_df)
    except Exception as e:
        print(f"FAILED: {e}")
    print()

Q: How many loads were delivered in the last full month available in the data?


--- SQL ---
SELECT COUNT(loadsmart_id) AS delivered_loads
FROM fact_loadsmart
WHERE load_was_cancelled = FALSE
  AND delivery_date >= DATE '2024-12-01' 
  AND delivery_date < DATE '2025-01-01';
 ---


,delivered_loads
0,442



Q: Which shipper had the highest total book price?


--- SQL ---
SELECT 
    s.shipper_name, 
    SUM(f.book_price) AS total_book_price
FROM 
    dim_shippers s
JOIN 
    fact_loadsmart f ON s.shipper_key = f.shipper_key
WHERE 
    f.load_was_cancelled = FALSE
GROUP BY 
    s.shipper_name
ORDER BY 
    total_book_price DESC
LIMIT 1;
 ---


...

,shipper_key,shipper_name,month,load_count,prev_month_load_count
0,be92243fcbb80d0da9e98f9fdd8a0a6b,Shipper 758,2024-11-01,192,<NA>



Q: Among lanes with at least 10 delivered loads, which had the highest average book price?


--- SQL ---
WITH delivered_loads AS (
    SELECT
        lane_key,
        AVG(book_price) AS avg_book_price
    FROM
        fact_loadsmart
    WHERE
        load_was_cancelled = FALSE
        AND delivery_date >= DATE '2024-12-01'
        AND delivery_date < DATE '2025-01-01'
    GROUP BY
        lane_key
    HAVING
        COUNT(loadsmart_id) >= 10
)
SELECT
    d.lane,
    d.avg_book_price
FROM
    delivered_loads dl
JOIN
    dim_lanes d ON dl.lane_key = d.lane_key
ORDER BY
    d.avg_book_price DESC
LIMIT 1;
 ---


--- retry SQL ---
WITH lane_loads AS (
    SELECT
        lane_key,
        AVG(book_price) AS avg_book_price,
        COUNT(*) AS load_count
    FROM
        fact_loadsmart
    WHERE
        load_was_cancelled = FALSE
    GROUP BY
        lane_key
    HAVING
        COUNT(*) >= 10
)
SELECT
    l.lane,
    l.pickup_city,
    l.pickup_state,
    l.delivery_city,
    l.delivery_state,
    d.avg_book_price
FROM
    dim_lanes l
JOIN
    lane_loads d ON l.lane_key = d.lane_key
ORDER BY
    d.avg_book_price DESC
LIMIT 1;
 ---


,lane,pickup_city,pickup_state,delivery_city,delivery_state,avg_book_price
0,"Stockton,CA -> Parrish,FL",Stockton,CA,Parrish,FL,6800.0
